In [ ]:
# =========================================================
# Import all dependencies
# =========================================================
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [ ]:
path_to_save = "/media3/majumder/astra/figures/"
path_to_load="/media3/majumder/emb/run_20251231_140014/umap_emb_all_dataset/umap_embeddings_finetuning.parquet"
df = pd.read_parquet(path_to_load, engine='pyarrow')
print("Pandas successfully read it!")
print(f"\n-- Total records: {df.shape[0]}; Total columns: {df.shape[1]}")

## Generate the hub of the UMAP plot

In [ ]:
label_to_superclass = {
    "DSCT|GDOR|SXPHE": "SC1", 
    "LPV": "SC1", 
    "RR": "SC1", 
    "CEP": "SC1",
    "AGN": "SC2",
    "SOLAR_LIKE": "SC3", 
    "RS": "SC3", 
    "ELL": "SC3",
    "ECL": "SC4",
    "S": "SC5", 
    "YSO": "SC5",
    "CV": "SC6"
}
superclass_colors = {
    "SC1": "#B95DFA", # Purple
    "SC2": "#FAF55D", # Yellow
    "SC3": "#5D62FA", # Blue
    "SC4": "#08EFF7", # Cyan
    "SC5": "#BBFA5D", # Green
    "SC6": "#FA6F5D"  # Red
}
# --------------------------------------------------------------------------------------
df = df[df['label'].isin(label_to_superclass.keys())].copy()
df['super_class'] = df['label'].map(label_to_superclass)
df['base_color'] = df['super_class'].map(superclass_colors)
# --------------------------------------------------------------------------------------

In [ ]:
def get_style(count):
    if count > 700_000:
        return 0.10, 0.5
    elif count > 500_000:
        return 0.10, 0.5
    elif count > 300_000:
        return 0.15, 1.0
    elif count > 200_000:
        return 0.30, 2.0
    elif count > 100_000:
        return 0.45, 2.5
    elif count > 80_000:
        return 0.45, 2.5
    elif count > 40_000:
        return 0.5, 3.0
    elif count > 20_000:
        return 0.5, 4.0
    elif count > 5_000:
        return 0.80, 7.0
    else:
        return 0.90, 8.0

In [ ]:
# ------------------------------------------------------------------------------------------------------------

df[['opacity', 'marker_size']] = df['total_records'].apply(lambda x: pd.Series(get_style(x)))

df['rgba_color'] = df.apply(
                            lambda row: mcolors.to_rgba(row['base_color'], alpha=row['opacity']), 
                            axis=1
                        )
# ------------------------------------------------------------------------------------------------------------
# df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
df_sorted = df.sort_values(by='total_records', ascending=False).reset_index(drop=True)
# ------------------------------------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(
                df_sorted['umap-x'], 
                df_sorted['umap-y'], 
                c=list(df_sorted['rgba_color']), s=df_sorted['marker_size'], edgecolors='none'
        )

ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)

output_filename = os.path.join(path_to_save, f'center_umap_superclasses_ft.png')
plt.tight_layout()
plt.savefig(
    output_filename, 
    dpi=600, 
    transparent=True, 
    bbox_inches='tight'
)
plt.show()
plt.close()

## Generate the inlet plots

In [ ]:
label_to_superclass = {
    "DSCT|GDOR|SXPHE": "SC1", 
    "LPV": "SC1", 
    "RR": "SC1", 
    "CEP": "SC1",
    "AGN": "SC2",
    "SOLAR_LIKE": "SC3", 
    "RS": "SC3", 
    "ELL": "SC3",
    "ECL": "SC4",
    "S": "SC5", 
    "YSO": "SC5",
    "CV": "SC6"
}

superclass_colors = {
    "SC1": "#B95DFA", # Purple
    "SC2": "#FAF55D", # Yellow
    "SC3": "#5D62FA", # Blue
    "SC4": "#08EFF7", # Cyan
    "SC5": "#BBFA5D", # Green
    "SC6": "#FA6F5D"  # Red
}
custom_inlet_palettes = {
    "SC1": ["#B708EC", "#D8A5F8", "#8906CB", "#2B0047"], 
    "SC3": ["#ABADF6", "#3437F8", "#050BB7"],                                 
}
# --------------------------------------------------------------------------------------
df = df[df['label'].isin(label_to_superclass.keys())].copy()
df['super_class'] = df['label'].map(label_to_superclass)
# --------------------------------------------------------------------------------------

In [ ]:
def get_shades(base_color, n):
    """Generates 'n' shades going from the base color to a darker hue."""
    if n == 1:
        return [base_color]
    palette = sns.dark_palette(base_color, n_colors=n+2, reverse=True)
    return [mcolors.to_hex(c) for c in palette[:n]]


def get_style(count):
    if count > 700_000:
        return 0.10, 0.5
    elif count > 500_000:
        return 0.10, 0.5
    elif count > 300_000:
        return 0.10, 0.5
    elif count > 200_000:
        return 0.10, 0.5
    elif count > 100_000:
        return 0.10, 0.5
    elif count > 80_000:
        return 0.40, 4.0
    elif count > 40_000:
        return 0.50, 4.5
    elif count > 20_000:
        return 0.50, 5.0
    elif count > 5_000:
        return 0.80, 6.0
    else:
        return 0.90, 8.0

In [ ]:
print("Generating 6 Inlet Plots...")
print("-" * 30)

for sc, base_color in superclass_colors.items():

    df_sc = df[df['super_class'] == sc].copy()
    
    subclasses = df_sc.groupby('label')['total_records'].first().sort_values(ascending=False).index.tolist()
    
    # ------------------------
    if sc in custom_inlet_palettes:
        shades = custom_inlet_palettes[sc][:len(subclasses)]
    else:
        shades = get_shades(base_color, len(subclasses))
    # ------------------------
    subclass_to_color = dict(zip(subclasses, shades))
    df_sc['sub_hex'] = df_sc['label'].map(subclass_to_color)
    
    print(f"{sc} Color Key:")
    for sub, hex_code in subclass_to_color.items():
        print(f"  {sub}: {hex_code}")
    print("-" * 30)
    
    # Apply opacity and size logic
    df_sc[['opacity', 'marker_size']] = df_sc['total_records'].apply(lambda x: pd.Series(get_style(x)))
    
    # Convert hex + opacity to RGBA
    df_sc['rgba_color'] = df_sc.apply(
                                        lambda row: mcolors.to_rgba(row['sub_hex'], alpha=row['opacity']), 
                                        axis=1
                                    )
    # Sort descending so the massive sub-classes go in the background
    df_sc_sorted = df_sc.sort_values(by='total_records', ascending=False).reset_index(drop=True)
    
    # Plotting
    fig, ax = plt.subplots(figsize=(5, 5)) # Slightly smaller figure size for inlets
    
    ax.scatter(
                df_sc_sorted['umap-x'], 
                df_sc_sorted['umap-y'], 
                c=list(df_sc_sorted['rgba_color']), 
                s=df_sc_sorted['marker_size'],
                edgecolors='none'
            )
    
    # Clean up the axes entirely
    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Export the inlet plot
    output_filename = os.path.join(path_to_save, f"inlet_{sc}_ft.png")
    plt.tight_layout()
    plt.savefig(output_filename, dpi=600, transparent=True, bbox_inches='tight')
    plt.show()
    plt.close() 

print("All 6 inlet plots successfully saved!")